<a href="https://colab.research.google.com/github/ShreyaCR/ai-art-mentor-agent-stepup/blob/main/agenticAIN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install openai duckduckgo-search

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 44.7 MB/s eta 0:00:00


In [2]:
import json

from openai import OpenAI
from duckduckgo_search import DDGS

In [3]:
import os
from getpass import getpass

OPENAI_API_KEY = getpass("Enter API key: ")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

client = OpenAI()

KeyboardInterrupt: Interrupted by user

In [3]:
def load_memory():

    try:
        with open("memory.json", "r") as f:
            return json.load(f)

    except:
        return []


def save_memory(memory):

    with open("memory.json", "w") as f:
        json.dump(memory, f)

In [5]:
def web_search(query):

    search_results = ""

    with DDGS() as ddgs:

        results = ddgs.text(
            query,
            max_results=3
        )

        for r in results:

            search_results += (
                r["title"]
                + "\n"
                + r["body"]
                + "\n\n"
            )

In [6]:
ART_MENTOR_PROMPT = """
You are an expert art mentor.

Help users with:

- Watercolor painting
- Composition
- Portraits
- Color mixing
- Creative ideas
- Artistic growth

Give practical advice.
Keep answers clear and encouraging.
"""

In [7]:
def ask_ai(user_input, memory):

    search_info = web_search(user_input)

    messages = [

        {
            "role": "system",
            "content": ART_MENTOR_PROMPT
        }

    ]

    messages.extend(memory)

    messages.append(

        {
            "role": "user",
            "content":
            f"""
User request:

{user_input}

Web information:

{search_info}
"""
        }

    )

    response = client.chat.completions.create(

        model="gpt-4.1-mini",

        messages=messages

    )

    return response.choices[0].message.content

In [8]:
memory = load_memory()

while True:

    user_input = input("You: ")

    if user_input.lower() == "exit":
        break

    reply = ask_ai(user_input, memory)

    print("\nAI:", reply)

    memory.append(
        {
            "role": "user",
            "content": user_input
        }
    )

KeyboardInterrupt: Interrupted by user

In [9]:
!pip install openai gradio pillow duckduckgo-search


In [10]:
import json

from PIL import Image

import gradio as gr

from openai import OpenAI

from duckduckgo_search import DDGS

In [11]:
import os
from getpass import getpass

OPENAI_API_KEY = getpass("Enter API key: ")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

client = OpenAI()

KeyboardInterrupt: Interrupted by user

In [12]:
def load_memory():

    try:

        with open("memory.json","r") as f:

            return json.load(f)

    except:

        return []


def save_memory(memory):

    with open("memory.json","w") as f:

        json.dump(memory,f)

In [13]:
def web_search(query):

    result_text = ""

    with DDGS() as ddgs:

        results = ddgs.text(
            query,
            max_results=3
        )

        for r in results:

            result_text += (
                r["title"]
                + "\n"
                + r["body"]
                + "\n\n"
            )

In [14]:
ART_MENTOR_PROMPT = """
You are a professional watercolor mentor.

When a painting is uploaded:

1. Analyze composition.
2. Analyze values.
3. Analyze edges.
4. Analyze color harmony.
5. Suggest improvements.
6. Suggest next washes.
7. Suggest glazing techniques.
8. Suggest reference ideas.

Be practical and encouraging.
"""

In [1]:
def generate_reference_prompt(user_request):

    response = client.chat.completions.create(

        model="gpt-4.1-mini",

        messages=[

            {
                "role":"system",
                "content":
                "Generate beautiful watercolor painting prompts."
            },

            {
                "role":"user",
                "content":user_request
            }

        ]

    )

    return response.choices[0].message.content


In [4]:
memory = load_memory()

def art_agent(text,image):

    web_info = web_search(text)

    if image is not None:

        answer = critique_painting(
            image,
            text
        )

    else:

        messages = [

            {
                "role":"system",
                "content":ART_MENTOR_PROMPT
            }

        ]

        messages.extend(memory)

        messages.append(

            {
                "role":"user",
                "content":
                f"""
Question:

{text}

Web:

{web_info}
"""
            }

        )

        response = client.chat.completions.create(

            model="gpt-4.1-mini",

            messages=messages

        )

        answer = response.choices[0].message.content

    memory.append(
        {
            "role":"user",
            "content":text
        }
    )